# 02 — Reward Hacking

Reward hacking occurs when an agent satisfies the reward function without fulfilling the intended objective.

## Goodhart's Law and Specification Gaming

**Specification gaming** (Krakovna et al., 2020): an agent finds unintended solutions that satisfy the reward function but not the designer's intent.

Examples from real RL systems:
- CoastRunners (OpenAI): agent learned to circle and collect fires instead of completing the race
- Robot hand: learned to flip its hand in the air for simulated grasping rewards rather than actually grasping
- Tetris: paused the game to avoid losing, since negative reward came from game over

**Reward hacking vs reward misspecification**:
- *Misspecification*: the reward is wrong by design (designer made a mistake)
- *Hacking*: the reward is a proxy for the true objective, and the agent exploits the gap

**Why it's hard to fix**:
1. Powerful optimisers are better at finding reward hacks
2. More compute = more exploitation of reward loopholes
3. Hacks are often valid in-distribution but fail catastrophically out-of-distribution

**Mitigation strategies**: reward modelling from human preferences, regularisation toward a safe prior, specification of reward using formal logic, reward learning from demonstrations.

In [ ]:
# Reward hacking detection
import numpy as np
import matplotlib.pyplot as plt

# Simulate an RL environment where agents can hack the reward
class CoastRunnerEnv:
    """
    Simplified CoastRunners: agent can race (intended) or collect fires (hack).
    True objective: complete the race
    Proxy reward: score (points for position + fires)
    """
    def __init__(self, n_steps=100):
        self.n_steps = n_steps

    def simulate_agent(self, hack_tendency=0.0):
        """
        hack_tendency: 0=races honestly, 1=maximally hacks reward.
        Returns: proxy_score, race_completion, is_hacking
        """
        np.random.seed(int(hack_tendency * 1000))
        fire_score = hack_tendency * 150 + np.random.randn() * 10  # high if hacking
        race_score = (1 - hack_tendency) * 80 + np.random.randn() * 5
        proxy_score = fire_score + race_score
        race_completion = (1 - hack_tendency) + np.random.randn() * 0.05
        return proxy_score, np.clip(race_completion, 0, 1)

hack_tendencies = np.linspace(0, 1, 20)
proxy_scores, race_completions = [], []

for ht in hack_tendencies:
    ps, rc = CoastRunnerEnv().simulate_agent(ht)
    proxy_scores.append(ps)
    race_completions.append(rc)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(hack_tendencies, proxy_scores, 'o-', color='steelblue')
ax1.set_xlabel('Hack tendency')
ax1.set_ylabel('Proxy score (reward)')
ax1.set_title('Proxy Reward vs Hack Tendency')

ax2.plot(proxy_scores, race_completions, 'o-', color='tomato')
ax2.set_xlabel('Proxy score (reward)')
ax2.set_ylabel('True objective (race completion)')
ax2.set_title('Goodhart Gap: High Reward ≠ True Success')

plt.suptitle('Reward Hacking in CoastRunners')
plt.tight_layout()
plt.savefig('/tmp/reward_hacking.png', dpi=100, bbox_inches='tight')
plt.show()

# Hacking detector
def detect_reward_hacking(proxy_scores, true_scores, threshold=0.3):
    """
    Flag if correlation between proxy and true objective is low.
    """
    corr = np.corrcoef(proxy_scores, true_scores)[0, 1]
    return corr < threshold, corr

is_hacking, corr = detect_reward_hacking(proxy_scores, race_completions)
print(f'Reward hacking detected: {is_hacking}')
print(f'Proxy-objective correlation: {corr:.3f}')
print('(Low correlation indicates the proxy is being gamed rather than the true objective)')